# Avaliar NOSSAS imagens com o modelo one-class (PatchCore)

Usa o **PatchCore treinado no MVTecAD bottle** (ckpt em `/home/<usuario>/results`) para pontuar as **nossas imagens** (padrão `dataset/`).

**AVISO honesto:** o modelo foi treinado no MVTec (perspectiva/iluminação diferentes). Score alto = **diferença de domínio**, não defeito. Isto é um sanity check de inferência; a validação real exige treinar no **nosso dataset**.

In [1]:
import glob, shutil
from pathlib import Path
from anomalib.data import Folder
from anomalib.engine import Engine
from anomalib.models import Patchcore

# AJUSTE aqui: pasta das NOSSAS imagens
OUR = Path("/home/<usuario>/tcc-pnaat/github/dataset")
MVTEC_GOOD = Path("/home/<usuario>/tcc-pnaat/datasets/MVTecAD/bottle/train/good")
STAGING = Path.home() / "tcc-pnaat/datasets/nossas"

good, bad = STAGING/"good", STAGING/"bad"
good.mkdir(parents=True, exist_ok=True); bad.mkdir(parents=True, exist_ok=True)
for d in (good,bad):
    for f in d.iterdir():
        if f.is_file(): f.unlink()
for f in sorted(MVTEC_GOOD.glob("*.png"))[:3]:      # referencia "normal"
    shutil.copy(f, good/f.name)
n=0
for f in sorted(OUR.glob("*.jpg")):                 # NOSSAS imagens
    shutil.copy(f, bad/f.name); n+=1
print("imagens nossas:", n)

/home/<usuario>/tcc-pnaat/github/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/<usuario>/tcc-pnaat/github/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/<usuario>/tcc-pnaat/github/.venv/lib/python3.11/site-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


imagens nossas: 81


In [2]:
ckpt = sorted(glob.glob(str(Path.home()/"results/Patchcore/MVTecAD/bottle/*/weights/lightning/model.ckpt")))[-1]
dm = Folder(name="nossas", root=str(STAGING), normal_dir="good", abnormal_dir="bad")
dm.setup()
engine = Engine(enable_progress_bar=False)
preds = engine.predict(model=Patchcore(), datamodule=dm, ckpt_path=ckpt, return_predictions=True)

rows = []
for pb in preds:
    paths = pb.image_path if isinstance(pb.image_path, (list, tuple)) else [pb.image_path]
    scs = pb.pred_score.detach().cpu().numpy().reshape(-1) if hasattr(pb.pred_score, "detach") else pb.pred_score
    if not hasattr(scs, "__len__"):
        scs = [scs]
    for i, path in enumerate(paths):
        name = Path(path).name
        sc = float(scs[i]) if i < len(scs) else (float(scs[0]) if len(scs) else 0.0)
        rows.append((name, sc))
rows.sort(key=lambda r: -r[1])
print("TOTAL", len(rows))
print("TOP 25 (maior anomaly score):")
for name, sc in rows[:25]:
    print(f"  {name}\t{sc:.4f}")
print("acima de 0.5:", sum(1 for _, s in rows if s > 0.5), "| abaixo de 0.3:", sum(1 for _, s in rows if s < 0.3))

Zero subset length encountered during splitting. This means one of your subsets
            might be empty or devoid of either normal or anomalous images.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Zero subset length encountered during splitting. This means one of your subsets
            might be empty or devoid of either normal or anomalous images.
Restoring states from the checkpoint path at /home/<usuario>/results/Patchcore/MVTecAD/bottle/v3/weights/lightning/model.ckpt
/home/<usuario>/tcc-pnaat/github/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/home/<usuario>/results/Patchcore/MVTecAD/bottle/v3/weights/lightning' to '/home/<usuario>/tcc-pnaat/github/

TOTAL 41
TOP 25 (maior anomaly score):
  frame_0002.jpg	1.0000
  frame_0003.jpg	1.0000
  frame_0004.jpg	1.0000
  frame_0005.jpg	1.0000
  frame_0007.jpg	1.0000
  frame_0009.jpg	1.0000
  frame_0012.jpg	1.0000
  frame_0014.jpg	1.0000
  frame_0016.jpg	1.0000
  frame_0017.jpg	1.0000
  frame_0018.jpg	1.0000
  frame_0022.jpg	1.0000
  frame_0024.jpg	1.0000
  frame_0027.jpg	1.0000
  frame_0028.jpg	1.0000
  frame_0031.jpg	1.0000
  frame_0032.jpg	1.0000
  frame_0033.jpg	1.0000
  frame_0039.jpg	1.0000
  frame_0041.jpg	1.0000
  frame_0042.jpg	1.0000
  frame_0044.jpg	1.0000
  frame_0045.jpg	1.0000
  frame_0049.jpg	1.0000
  frame_0050.jpg	1.0000
acima de 0.5: 41 | abaixo de 0.3: 0


## Como ler
- Score alto = difere do padrão MVTec bottle (provável domínio).
- Para score de defeito real: treinar no **nosso dataset** (normal = nossas frames; defeitos = gerados pelo prompt) e repetir este fluxo no split do nosso dataset.